# Decoder-specific DEM prior calibration

This walkthrough fits known-support covariance priors and trains separate effective priors for MWPM and ldpc Union-find. Detector supports and logical actions stay fixed. Training uses observed detection events and logical outcomes. Independent validation shots select checkpoints, and a shared held-out test set measures the probability of an unsuccessful complete memory shot.

Install the project and its optional extras in the notebook kernel environment with `python -m pip install -e '.[decoder-training,notebook]'`. The default settings use the pinned THeatA checkout in `vendor/theata` and the smoke YAML. The research YAML is an editable larger example.

In [ ]:
# EDITABLE PARAMETERS
# Replace None with a value to override the selected YAML or package default.
from dataclasses import asdict
from pathlib import Path

from theata_prior.decoder_training.config import config_from_dict, load_training_config
from theata_prior.decoder_training.experiment import build, evaluate, fit, preflight, sample, train
from theata_prior.decoder_training.plotting import PlotOptions, load_results, plot_results

root = Path.cwd()
if not (root / "pyproject.toml").exists():
    root = root.parent
config_path = root / "configs/decoder_training_smoke.yaml"
output = root / "results/decoder_training_notebook"
resume = True
stages = ("build", "sample", "fit", "train", "evaluate", "plot")
timing_probe_shots = 128  # Set to zero to skip. Runs after sampling.

physical = dict(
    theata_path=None, hardware=None, distances=None, basis=None, physical_error_rates=None,
    sample_source=None, training_shots=None, test_shots=None, repetitions=None, seed=None,
    sample_chunk_shots=None, decode_chunk_shots=None, covariance_chunk_shots=None,
    covariance_block_detectors=None, probability_floor=None, probability_ceiling=None,
    strict_estimation=None,
)
experiment = dict(
    rounds_by_distance=None, validation_shots=None, optimizer_seeds=None, initializations=None,
    covariance_failure=None, uniform_probability=None, p_min=None, p_max=None,
    grouping=None, manual_groups=None, target_decoders=None, evaluation_decoders=None,
    ldpc_version=None, uf_mode=None, bits_per_step=None,
)
cem = dict(
    population_size=None, elite_count=None, generations=None, initial_std=None, smoothing=None,
    min_std=None, theta_bounds=None, train_batch_shots=None, validation_interval=None,
)
use_full_training_prefix = False  # True sets train_batch_shots to null.
resources = dict(
    total_workers=None, case_jobs=None, candidate_jobs=None, sampling_jobs=None,
    covariance_jobs=None, native_threads=None,
)
plots = PlotOptions(
    distance=None, physical_error_rate=None, calibration_shots=None, replicate=None,
    optimizer_seed=None, initialization=None, color_normalization="linear", formats=("png", "pdf"),
)

## Validate settings and estimate resource use

Preflight can count the sector detectors from the ideal circuit. It reports the exact mechanism and group counts after the physical models are built.

In [ ]:
from IPython.display import Image, Markdown, display
from theata_prior.io import read_json

settings = asdict(load_training_config(config_path))
for section, overrides in (("physical", physical), ("cem", cem), ("resources", resources)):
    settings[section].update({key: value for key, value in overrides.items() if value is not None})
settings.update({key: value for key, value in experiment.items() if value is not None})
if use_full_training_prefix:
    settings["cem"]["train_batch_shots"] = None
config = config_from_dict(settings, config_path)
if not resume and any(stage != "plot" for stage in stages) and (output / "decoder_training.json").exists():
    raise FileExistsError(f"Choose a fresh output directory or enable resume: {output}")

def table(rows, columns):
    def display_value(value):
        if isinstance(value, float):
            return f"{value:.4g}"
        return "—" if value is None else str(value)
    lines = ["| " + " | ".join(columns) + " |", "| " + " | ".join(["---"] * len(columns)) + " |"]
    lines += ["| " + " | ".join(display_value(row.get(column)) for column in columns) + " |" for row in rows]
    display(Markdown("\n".join(lines)))

display(preflight(config, output))

## Build the physical models and sample independent streams

Training budgets use nested prefixes of the same training pool. Validation shots are additional to the calibration budget. All methods in a physical case use the same test samples.

In [ ]:
if "build" in stages:
    built = build(config, output)
    display(preflight(config, output))
if "sample" in stages:
    sample(config, output)
    if timing_probe_shots:
        display(preflight(config, output, probe_shots=timing_probe_shots))

## Fit covariance baselines and inspect groups

The known-support fit uses only training detections. Its regularized probabilities are kept alongside the signed raw pair estimates, validity mask, and empirical detector covariance. Group features come from verified detector coordinates or the detector target count.

In [ ]:
if "fit" in stages:
    fits = fit(config, output)
    table(fits, ("distance", "physical_error_rate", "replicate", "calibration_shots", "status",
                 "raw_finite_pairs", "total_unique_pairs", "raw_negative_pairs"))
if "build" in stages:
    groups = read_json(built[0].directory / "groups.json")
    display(groups["features"])
    table([{"group": label, "mechanisms": size, "reference": index == groups["reference"]}
           for index, (label, size) in enumerate(zip(groups["labels"], groups["sizes"]))],
          ("group", "mechanisms", "reference"))

## Train, freeze, and evaluate

CEM adjusts group weights at a fixed mean LLR scale. Its selected priors are effective decoder parameters. The prior error tables measure their separate agreement with the actual canonical mechanism probabilities. A flat training run or an initialization selected by validation is a valid outcome.

In [ ]:
if "train" in stages:
    training = train(config, output)
    table(training, ("target_decoder", "status", "optimizer_evaluations", "validation_evaluations",
                     "decoder_shot_evaluations", "distinct_train_shots", "flat_generations"))
if "evaluate" in stages:
    evaluate(config, output)
if (output / "evaluation_complete.json").exists():
    evaluation, prior_errors, artifact_index = load_results(output)
    table(evaluation[:20], ("method", "evaluation_decoder", "status", "test_shots", "failures",
                            "decoder_failures", "failure_probability", "ci_low", "ci_high"))
    table([row for row in prior_errors if row["comparison"] == "actual" and row["metric"] in
           ("mechanisms", "pair_mechanisms", "singleton_mechanisms")][:20],
          ("method", "metric", "status", "rmse", "mae", "count"))

## Plot from saved results

The detailed panels use the selected data replicate and optimizer seed. Sweep figures retain the individual models with fixed-model Wilson intervals. The separate variation table describes spread across replicates and optimizer seeds while retaining the shared test design. Set `stages = ("plot",)` to reload and replot an existing run.

In [ ]:
if "plot" in stages:
    reloaded = load_results(output)
    figure_paths = plot_results(output, plots)
    display(Markdown(f"Loaded {len(reloaded[0])} saved evaluation rows and generated "
                     f"{len(figure_paths)} figures in {output / 'figures'}."))
    for path in figure_paths:
        if path.suffix == ".png" and path.stem in {
            "pair_priors", "pair_residuals_vs_actual", "raw_covariance_pair_priors",
            "cross_decoder_transfer", "failure_vs_physical_error", "failure_vs_calibration_shots",
        }:
            display(Image(filename=str(path)))